# Paper Tables

Generate LaTeX tables for the paper.

In [33]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

## Helper Functions

In [34]:
def load_run(fpath):
    """Load results from a single run."""
    fpath = Path(fpath)
    if not fpath.exists():
        return None
    with open(fpath) as f:
        return [json.loads(l) for l in f]

def compute_unified_metrics(entries):
    """Compute metrics for unified approach (baseline, directed, nudged)."""
    total = len(entries)
    compiled = sum(1 for e in entries if e.get("success") or e.get("lean_verification", {}).get("success", False))
    correct = sum(1 for e in entries if e.get("correct"))
    correct_compiled = sum(1 for e in entries 
                          if (e.get("success") or e.get("lean_verification", {}).get("success", False)) 
                          and e.get("correct"))
    return {
        "total": total,
        "compiled": compiled,
        "correct": correct,
        "correct_compiled": correct_compiled,
        "compilation": compiled / total * 100 if total > 0 else 0,
        "accuracy_overall": correct / total * 100 if total > 0 else 0,
        "accuracy_compiled": correct_compiled / compiled * 100 if compiled > 0 else 0,
    }

def compute_twostage_metrics(entries):
    """Compute metrics for two-stage approach."""
    total = len(entries)
    s1_pass = sum(1 for e in entries if e.get("stage1_success"))
    s2_pass = sum(1 for e in entries if e.get("stage2_success"))
    correct = sum(1 for e in entries if e.get("correct"))
    correct_compiled = sum(1 for e in entries if e.get("stage2_success") and e.get("correct"))
    return {
        "total": total,
        "compiled": s2_pass,
        "correct": correct,
        "correct_compiled": correct_compiled,
        "stage1": s1_pass / total * 100 if total > 0 else 0,
        "stage2": s2_pass / total * 100 if total > 0 else 0,
        "accuracy_overall": correct / total * 100 if total > 0 else 0,
        "accuracy_compiled": correct_compiled / s2_pass * 100 if s2_pass > 0 else 0,
    }

def fmt(values):
    """Format mean±std for LaTeX."""
    mean = np.mean(values)
    std = np.std(values)
    return f"{mean:.1f}\\tiny{{±{std:.1f}}}"

# Configuration
CONDITIONS = [
    ("Baseline", "baseline"),
    ("Directed$_\\text{T}$", "bidir_true"),
    ("Directed$_\\text{F}$", "bidir_false"),
    ("Nudged$_\\text{T}$", "spooky_true"),
    ("Nudged$_\\text{F}$", "spooky_false"),
]

MODELS = [("GPT-5", "gpt-5"), ("DeepSeek-R1", "deepseek")]
DATASETS = [("FOLIO", "folio"), ("M-LogiEval", "multilogieval")]

# Overall

## Table 1: FOLIO Results (mean±std over 3 runs)

- **Acc (Overall)**: correct / total (all cases)
- **Acc (Compiled)**: correct_compiled / compiled (only verified cases)

In [35]:
# FOLIO Results with both accuracy metrics
print("FOLIO RESULTS (mean±std over 3 runs)")
print("=" * 95)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Acc (Overall)':>15} {'Acc (Compiled)':>16}")
print("-" * 95)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        folio_comp, folio_acc_overall, folio_acc_compiled = [], [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/folio/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                folio_comp.append(m["compilation"])
                folio_acc_overall.append(m["accuracy_overall"])
                folio_acc_compiled.append(m["accuracy_compiled"])
        
        fc = f"{np.mean(folio_comp):.1f}±{np.std(folio_comp):.1f}"
        fa_o = f"{np.mean(folio_acc_overall):.1f}±{np.std(folio_acc_overall):.1f}"
        fa_c = f"{np.mean(folio_acc_compiled):.1f}±{np.std(folio_acc_compiled):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {fc:>15} {fa_o:>15} {fa_c:>16}")
    
    # Two-Stage (show S1 and S2 compilation)
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    folio_s1, folio_s2, folio_acc_overall, folio_acc_compiled = [], [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/folio/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            folio_s1.append(m["stage1"])
            folio_s2.append(m["stage2"])
            folio_acc_overall.append(m["accuracy_overall"])
            folio_acc_compiled.append(m["accuracy_compiled"])
    
    fs1 = f"{np.mean(folio_s1):.1f}±{np.std(folio_s1):.1f}"
    fs2 = f"{np.mean(folio_s2):.1f}±{np.std(folio_s2):.1f}"
    fa_o = f"{np.mean(folio_acc_overall):.1f}±{np.std(folio_acc_overall):.1f}"
    fa_c = f"{np.mean(folio_acc_compiled):.1f}±{np.std(folio_acc_compiled):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{fs1} S2:{fs2}  {fa_o:>10} {fa_c:>16}")
    print()

FOLIO RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %   Acc (Overall)   Acc (Compiled)
-----------------------------------------------------------------------------------------------
GPT-5          Baseline                    98.2±0.8        85.2±0.4         85.3±0.9
GPT-5          Directed$_\text{T}$         99.7±0.5        91.3±0.2         91.3±0.2
GPT-5          Directed$_\text{F}$         99.2±0.5        92.0±0.5         91.9±0.4
GPT-5          Nudged$_\text{T}$           99.5±0.4        92.1±0.4         92.1±0.4
GPT-5          Nudged$_\text{F}$           98.9±0.6        92.3±0.2         92.5±0.4
GPT-5          Two-Stage            S1:100.0±0.0 S2:81.6±1.9    72.7±4.3         69.9±4.7

DeepSeek-R1    Baseline                    94.6±1.1        86.4±0.6         86.6±1.2
DeepSeek-R1    Directed$_\text{T}$         95.1±1.1        91.0±0.5         91.2±0.5
DeepSeek-R1    Directed$_\text{F}$         92.1±1.5        91.1±0.4         91.8±0.8
DeepSeek-R1

## Table 2: Multi-LogiEval Results (mean±std over 3 runs)

- **Acc (Overall)**: correct / total (all cases)
- **Acc (Compiled)**: correct_compiled / compiled (only verified cases)

In [36]:
# Multi-LogiEval Results with both accuracy metrics
print("MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)")
print("=" * 95)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Acc (Overall)':>15} {'Acc (Compiled)':>16}")
print("-" * 95)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        multi_comp, multi_acc_overall, multi_acc_compiled = [], [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/multilogieval/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                multi_comp.append(m["compilation"])
                multi_acc_overall.append(m["accuracy_overall"])
                multi_acc_compiled.append(m["accuracy_compiled"])
        
        mc = f"{np.mean(multi_comp):.1f}±{np.std(multi_comp):.1f}"
        ma_o = f"{np.mean(multi_acc_overall):.1f}±{np.std(multi_acc_overall):.1f}"
        ma_c = f"{np.mean(multi_acc_compiled):.1f}±{np.std(multi_acc_compiled):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {mc:>15} {ma_o:>15} {ma_c:>16}")
    
    # Two-Stage (show S1 and S2 compilation)
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    multi_s1, multi_s2, multi_acc_overall, multi_acc_compiled = [], [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/multilogieval/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            multi_s1.append(m["stage1"])
            multi_s2.append(m["stage2"])
            multi_acc_overall.append(m["accuracy_overall"])
            multi_acc_compiled.append(m["accuracy_compiled"])
    
    ms1 = f"{np.mean(multi_s1):.1f}±{np.std(multi_s1):.1f}"
    ms2 = f"{np.mean(multi_s2):.1f}±{np.std(multi_s2):.1f}"
    ma_o = f"{np.mean(multi_acc_overall):.1f}±{np.std(multi_acc_overall):.1f}"
    ma_c = f"{np.mean(multi_acc_compiled):.1f}±{np.std(multi_acc_compiled):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{ms1} S2:{ms2}  {ma_o:>10} {ma_c:>16}")
    print()

MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %   Acc (Overall)   Acc (Compiled)
-----------------------------------------------------------------------------------------------
GPT-5          Baseline                    99.3±0.9        71.7±2.1         72.2±2.7
GPT-5          Directed$_\text{T}$         99.3±0.9        85.7±1.2         85.6±1.2
GPT-5          Directed$_\text{F}$         99.3±0.5        83.7±1.2         83.6±1.3
GPT-5          Nudged$_\text{T}$           99.0±0.8        92.0±0.8         92.3±0.9
GPT-5          Nudged$_\text{F}$           99.7±0.5        85.3±1.2         85.3±1.2
GPT-5          Two-Stage            S1:100.0±0.0 S2:89.7±4.5    56.7±2.9         59.1±3.1

DeepSeek-R1    Baseline                    97.3±0.9        69.7±1.7         70.5±2.1
DeepSeek-R1    Directed$_\text{T}$         96.3±2.5        82.3±2.5         82.6±3.1
DeepSeek-R1    Directed$_\text{F}$         94.0±2.2        82.0±1.6         82.6±1.4
De

# Conservative vs Definite Analysis

In [38]:
# Conservative vs Definite predictions analysis (COMPILED CASES ONLY) - mean±std
print("CONSERVATIVE VS DEFINITE PREDICTIONS (Compiled Cases Only)")
print("=" * 140)
print()
print("Conservative = Predicted Uncertain/Failure | Definite = Predicted True/False")
print()

CONDITIONS_ANALYSIS = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

for dataset, dataset_dir in DATASETS:
    print(f"\n{'='*140}")
    print(f"DATASET: {dataset}")
    print(f"{'='*140}")
    print(f"{'Model':<14} {'Condition':<12} {'Cons%':>12} {'ConsPrec':>12} {'ConsRec':>12} {'DefPrec':>12} {'DefRec':>12}")
    print("-" * 90)
    
    for model, model_dir in MODELS:
        for cond_name, cond in CONDITIONS_ANALYSIS:
            cons_pcts, cons_precs, cons_recs, def_precs, def_recs = [], [], [], [], []
            
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue
                
                compiled = 0
                conservative = 0
                cons_correct = 0
                definite = 0
                def_correct = 0
                gt_uncertain = 0
                gt_definite = 0
                
                for e in entries:
                    lean_success = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    if not lean_success:
                        continue
                    
                    compiled += 1
                    pred = str(e.get("prediction", "")).lower()
                    gt = str(e.get("ground_truth", "")).lower()
                    
                    if pred in ["uncertain", "unknown", "failure"]: pred = "uncertain"
                    if gt in ["uncertain", "unknown"]: gt = "uncertain"
                    if pred in ["yes"]: pred = "true"
                    if pred in ["no"]: pred = "false"
                    if gt in ["yes"]: gt = "true"
                    if gt in ["no"]: gt = "false"
                    
                    if gt == "uncertain":
                        gt_uncertain += 1
                    else:
                        gt_definite += 1
                    
                    if pred == "uncertain":
                        conservative += 1
                        if gt == "uncertain":
                            cons_correct += 1
                    else:
                        definite += 1
                        if pred == gt:
                            def_correct += 1
                
                if compiled > 0:
                    cons_pcts.append(conservative / compiled * 100)
                    cons_precs.append(cons_correct / conservative * 100 if conservative > 0 else 0)
                    cons_recs.append(cons_correct / gt_uncertain * 100 if gt_uncertain > 0 else 0)
                    def_precs.append(def_correct / definite * 100 if definite > 0 else 0)
                    def_recs.append(def_correct / gt_definite * 100 if gt_definite > 0 else 0)
            
            if cons_pcts:
                cp = f"{np.mean(cons_pcts):.1f}±{np.std(cons_pcts):.1f}"
                cpr = f"{np.mean(cons_precs):.1f}±{np.std(cons_precs):.1f}"
                cr = f"{np.mean(cons_recs):.1f}±{np.std(cons_recs):.1f}"
                dpr = f"{np.mean(def_precs):.1f}±{np.std(def_precs):.1f}"
                dr = f"{np.mean(def_recs):.1f}±{np.std(def_recs):.1f}"
                print(f"{model:<14} {cond_name:<12} {cp:>12} {cpr:>12} {cr:>12} {dpr:>12} {dr:>12}")
        
        # Two-Stage
        model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
        cons_pcts, cons_precs, cons_recs, def_precs, def_recs = [], [], [], [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue
            
            compiled = 0
            conservative = 0
            cons_correct = 0
            definite = 0
            def_correct = 0
            gt_uncertain = 0
            gt_definite = 0
            
            for e in entries:
                if not e.get("stage2_success", False):
                    continue
                
                compiled += 1
                pred = str(e.get("prediction", "")).lower()
                gt = str(e.get("ground_truth", "")).lower()
                
                if pred in ["uncertain", "unknown", "failure"]: pred = "uncertain"
                if gt in ["uncertain", "unknown"]: gt = "uncertain"
                if pred in ["yes"]: pred = "true"
                if pred in ["no"]: pred = "false"
                if gt in ["yes"]: gt = "true"
                if gt in ["no"]: gt = "false"
                
                if gt == "uncertain":
                    gt_uncertain += 1
                else:
                    gt_definite += 1
                
                if pred == "uncertain":
                    conservative += 1
                    if gt == "uncertain":
                        cons_correct += 1
                else:
                    definite += 1
                    if pred == gt:
                        def_correct += 1
            
            if compiled > 0:
                cons_pcts.append(conservative / compiled * 100)
                cons_precs.append(cons_correct / conservative * 100 if conservative > 0 else 0)
                cons_recs.append(cons_correct / gt_uncertain * 100 if gt_uncertain > 0 else 0)
                def_precs.append(def_correct / definite * 100 if definite > 0 else 0)
                def_recs.append(def_correct / gt_definite * 100 if gt_definite > 0 else 0)
        
        if cons_pcts:
            cp = f"{np.mean(cons_pcts):.1f}±{np.std(cons_pcts):.1f}"
            cpr = f"{np.mean(cons_precs):.1f}±{np.std(cons_precs):.1f}"
            cr = f"{np.mean(cons_recs):.1f}±{np.std(cons_recs):.1f}"
            dpr = f"{np.mean(def_precs):.1f}±{np.std(def_precs):.1f}"
            dr = f"{np.mean(def_recs):.1f}±{np.std(def_recs):.1f}"
            print(f"{model:<14} {'Two-Stage':<12} {cp:>12} {cpr:>12} {cr:>12} {dpr:>12} {dr:>12}")
        print()

print("\nLegend (all based on compiled cases only, mean±std over 3 runs):")
print("  Cons% = Conservative ratio (predicted Uncertain/Failure / compiled)")
print("  ConsPrec = Precision: correct Cons / predicted Cons")
print("  ConsRec = Recall: correct Cons / GT=Uncertain")
print("  DefPrec = Precision: correct Def / predicted Def")
print("  DefRec = Recall: correct Def / GT=Definite")

CONSERVATIVE VS DEFINITE PREDICTIONS (Compiled Cases Only)

Conservative = Predicted Uncertain/Failure | Definite = Predicted True/False


DATASET: FOLIO
Model          Condition           Cons%     ConsPrec      ConsRec      DefPrec       DefRec
------------------------------------------------------------------------------------------
GPT-5          Baseline         43.1±1.0     74.1±1.5     93.6±0.6     93.8±0.1     81.0±1.7
GPT-5          Directed_T       65.7±0.3     48.4±0.4     93.2±0.7     88.9±0.5     46.2±0.0
GPT-5          Directed_F       72.2±0.8     45.2±0.3     96.1±0.7     89.9±0.6     37.8±0.7
GPT-5          Nudged_T         61.7±0.1     50.0±0.3     90.8±0.7     86.2±0.6     50.0±0.3
GPT-5          Nudged_F         67.9±0.2     46.2±0.3     91.8±0.6     86.0±0.1     41.9±0.3
GPT-5          Two-Stage        26.9±8.6     71.3±4.5    51.7±17.1     70.5±7.4     80.8±2.8

DeepSeek-R1    Baseline         42.9±2.3     77.1±2.2     94.0±1.3     93.9±0.3     82.6±2.7
DeepSeek-R

# Iteration Analysis

In [ ]:
# Iteration analysis: iteration count, answer changes, and error rate correlation (COMPILED ONLY)
from collections import Counter

print("ITERATION ANALYSIS - COMPILED CASES ONLY (mean±std over 3 runs)")
print("=" * 140)

CONDITIONS_ITER = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

for dataset, dataset_dir in DATASETS:
    print(f"\n{'='*140}")
    print(f"DATASET: {dataset}")
    print(f"{'='*140}")

    for model, model_dir in MODELS:
        print(f"\n{model}")
        print("-" * 130)
        print(f"{'Condition':<12} {'Avg Iter':>10} {'Ans Chg%':>10} {'Err@1 (k)':>18} {'Err@2 (k)':>18} {'Err@3 (k)':>18}")
        print("-" * 130)

        for cond_name, cond in CONDITIONS_ITER:
            avg_iters, ans_chg_pcts = [], []
            err1s, err2s, err3s = [], [], []
            k1s, k2s, k3s = [], [], []

            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue

                iter_counts = []
                answer_changed = 0
                total_cases = 0
                errors_by_iter = {1: 0, 2: 0, 3: 0}
                total_by_iter = {1: 0, 2: 0, 3: 0}

                for e in entries:
                    lean_success = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    if not lean_success:
                        continue

                    iterations = e.get("iterations", [])
                    num_iter = e.get("num_iterations", len(iterations))
                    if num_iter == 0:
                        continue

                    total_cases += 1
                    iter_counts.append(num_iter)

                    if len(iterations) > 1:
                        preds = [it.get("prediction", "") for it in iterations if it.get("prediction")]
                        if len(set(preds)) > 1:
                            answer_changed += 1

                    if num_iter in total_by_iter:
                        total_by_iter[num_iter] += 1
                        if not e.get("correct", False):
                            errors_by_iter[num_iter] += 1

                if total_cases > 0:
                    avg_iters.append(np.mean(iter_counts))
                    ans_chg_pcts.append(answer_changed / total_cases * 100)
                    err1s.append(errors_by_iter[1] / total_by_iter[1] * 100 if total_by_iter[1] > 0 else np.nan)
                    err2s.append(errors_by_iter[2] / total_by_iter[2] * 100 if total_by_iter[2] > 0 else np.nan)
                    err3s.append(errors_by_iter[3] / total_by_iter[3] * 100 if total_by_iter[3] > 0 else np.nan)
                    k1s.append(total_by_iter[1])
                    k2s.append(total_by_iter[2])
                    k3s.append(total_by_iter[3])

            if avg_iters:
                ai = f"{np.mean(avg_iters):.2f}±{np.std(avg_iters):.2f}"
                ac = f"{np.mean(ans_chg_pcts):.1f}±{np.std(ans_chg_pcts):.1f}"
                e1 = f"{np.nanmean(err1s):.1f}% ({int(np.mean(k1s))})" if not all(np.isnan(err1s)) else "N/A"
                e2 = f"{np.nanmean(err2s):.1f}% ({int(np.mean(k2s))})" if not all(np.isnan(err2s)) else "N/A"
                e3 = f"{np.nanmean(err3s):.1f}% ({int(np.mean(k3s))})" if not all(np.isnan(err3s)) else "N/A"
                print(f"{cond_name:<12} {ai:>10} {ac:>10} {e1:>18} {e2:>18} {e3:>18}")

        # Two-Stage
        model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
        avg_s1_iters, avg_s2_iters, s2_chg_pcts = [], [], []
        s2_err1s, s2_err2s, s2_err3s = [], [], []
        s2_k1s, s2_k2s, s2_k3s = [], [], []

        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue

            s1_iters, s2_iters = [], []
            s2_changed = 0
            total_cases = 0
            errors_by_iter = {1: 0, 2: 0, 3: 0}
            total_by_iter = {1: 0, 2: 0, 3: 0}

            for e in entries:
                if not e.get("stage2_success", False):
                    continue

                s1_it = e.get("stage1_iterations", [])
                s2_it = e.get("stage2_iterations", [])

                total_cases += 1
                s1_num = len(s1_it) if s1_it else 1
                s2_num = len(s2_it) if s2_it else 1
                s1_iters.append(s1_num)
                s2_iters.append(s2_num)

                if s2_it and len(s2_it) > 1:
                    preds = [it.get("prediction", "") for it in s2_it if it.get("prediction")]
                    if len(set(preds)) > 1:
                        s2_changed += 1

                if s2_num in total_by_iter:
                    total_by_iter[s2_num] += 1
                    if not e.get("correct", False):
                        errors_by_iter[s2_num] += 1

            if total_cases > 0:
                avg_s1_iters.append(np.mean(s1_iters))
                avg_s2_iters.append(np.mean(s2_iters))
                s2_chg_pcts.append(s2_changed / total_cases * 100)
                s2_err1s.append(errors_by_iter[1] / total_by_iter[1] * 100 if total_by_iter[1] > 0 else np.nan)
                s2_err2s.append(errors_by_iter[2] / total_by_iter[2] * 100 if total_by_iter[2] > 0 else np.nan)
                s2_err3s.append(errors_by_iter[3] / total_by_iter[3] * 100 if total_by_iter[3] > 0 else np.nan)
                s2_k1s.append(total_by_iter[1])
                s2_k2s.append(total_by_iter[2])
                s2_k3s.append(total_by_iter[3])

        if avg_s1_iters:
            s1 = f"{np.mean(avg_s1_iters):.2f}±{np.std(avg_s1_iters):.2f}"
            s2 = f"{np.mean(avg_s2_iters):.2f}±{np.std(avg_s2_iters):.2f}"
            ai = f"S1:{s1} S2:{s2}"
            ac = f"{np.mean(s2_chg_pcts):.1f}±{np.std(s2_chg_pcts):.1f}"
            e1 = f"{np.nanmean(s2_err1s):.1f}% ({int(np.mean(s2_k1s))})" if not all(np.isnan(s2_err1s)) else "N/A"
            e2 = f"{np.nanmean(s2_err2s):.1f}% ({int(np.mean(s2_k2s))})" if not all(np.isnan(s2_err2s)) else "N/A"
            e3 = f"{np.nanmean(s2_err3s):.1f}% ({int(np.mean(s2_k3s))})" if not all(np.isnan(s2_err3s)) else "N/A"
            print(f"{'Two-Stage':<12} {ai:>24} {ac:>10} {e1:>18} {e2:>18} {e3:>18}")

print("\n" + "=" * 140)
print("Legend (compiled cases only):")
print("  Avg Iter = Average number of iterations needed (Two-Stage shows S1 and S2 separately)")
print("  Ans Chg% = % of cases where answer changed across iterations")
print("  Err@N (k) = Error rate for cases that took N iterations, k = avg sample size per run")

# Error Analysis

Structured error breakdown by GT → Pred for each condition.

In [40]:
# Structured error table - all wrong predictions by GT -> Pred, BY DATASET and MODEL

# Simple conditions (for display purposes)
CONDITIONS_SIMPLE = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

all_wrong = []
all_compiled = []
all_total = []

# Regular conditions
for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        for cond_name, cond in CONDITIONS_SIMPLE:
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue
                for e in entries:
                    all_total.append({"dataset": dataset, "model": model, "condition": cond_name})
                    lean_success = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    if lean_success:
                        all_compiled.append({"dataset": dataset, "model": model, "condition": cond_name})
                    if not lean_success or e.get("correct", False):
                        continue
                    pred = str(e.get("prediction", "")).lower()
                    gt = str(e.get("ground_truth", "")).lower()
                    if gt in ["yes"]: gt = "true"
                    elif gt in ["no"]: gt = "false"
                    if pred in ["yes"]: pred = "true"
                    elif pred in ["no"]: pred = "false"
                    all_wrong.append({"dataset": dataset, "model": model, "condition": cond_name, "gt": gt.capitalize(), "pred": pred.capitalize()})

# Two-stage
for dataset, dataset_dir in DATASETS:
    for model_name in ["gpt-5", "deepseek-r1"]:
        model_display = "GPT-5" if model_name == "gpt-5" else "DeepSeek-R1"
        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue
            for e in entries:
                all_total.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if e.get("stage2_success", False):
                    all_compiled.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if not e.get("stage2_success", False) or e.get("correct", False):
                    continue
                pred = str(e.get("prediction", "")).lower()
                gt = str(e.get("ground_truth", "")).lower()
                if gt in ["yes"]: gt = "true"
                elif gt in ["no"]: gt = "false"
                if pred in ["yes"]: pred = "true"
                elif pred in ["no"]: pred = "false"
                all_wrong.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage", "gt": gt.capitalize(), "pred": pred.capitalize()})

df_wrong_all = pd.DataFrame(all_wrong)
df_compiled_model = pd.DataFrame(all_compiled)
df_total_model = pd.DataFrame(all_total)

error_types = ["T→F", "T→Unc", "T→Fail", "F→T", "F→Unc", "F→Fail", "Unc→T", "Unc→F"]
error_map = {
    "T→F": ("True", "False"), "T→Unc": ("True", "Uncertain"), "T→Fail": ("True", "Failure"),
    "F→T": ("False", "True"), "F→Unc": ("False", "Uncertain"), "F→Fail": ("False", "Failure"),
    "Unc→T": ("Uncertain", "True"), "Unc→F": ("Uncertain", "False")
}

print("STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)")
print("="*100)
print(f"Total wrong (compiled): {len(df_wrong_all)}")
print()

for dataset in ["FOLIO", "M-LogiEval"]:
    print(f"\n{'='*100}")
    print(f"DATASET: {dataset}")
    print(f"{'='*100}")
    
    for model_name in ["GPT-5", "DeepSeek-R1"]:
        mask_wrong = (df_wrong_all["dataset"] == dataset) & (df_wrong_all["model"] == model_name)
        mask_comp = (df_compiled_model["dataset"] == dataset) & (df_compiled_model["model"] == model_name)
        mask_total = (df_total_model["dataset"] == dataset) & (df_total_model["model"] == model_name)
        
        model_wrong = df_wrong_all[mask_wrong]
        model_compiled = df_compiled_model[mask_comp]
        model_total = df_total_model[mask_total]
        
        print(f"\n{model_name}: {len(model_wrong)} wrong / {len(model_compiled)} compiled / {len(model_total)} total")
        print("-"*100)
        
        rows = []
        for cond in ["Baseline", "Directed_T", "Directed_F", "Nudged_T", "Nudged_F", "Two-Stage"]:
            cond_df = model_wrong[model_wrong["condition"] == cond]
            cond_compiled = len(model_compiled[model_compiled["condition"] == cond])
            cond_total = len(model_total[model_total["condition"] == cond])
            row = {"Condition": cond, "Wrong/Comp/Tot": f"{len(cond_df)}/{cond_compiled}/{cond_total}"}
            for etype in error_types:
                gt_val, pred_val = error_map[etype]
                count = len(cond_df[(cond_df["gt"] == gt_val) & (cond_df["pred"] == pred_val)])
                row[etype] = count if count > 0 else "-"
            rows.append(row)
        
        error_df = pd.DataFrame(rows)
        print(error_df.to_string(index=False))

print("\n" + "="*100)
print("Legend: T=True, F=False, Unc=Uncertain, Fail=Failure (couldn't prove)")
print("Wrong/Comp/Tot = Wrong among compiled / Compiled / Total cases")

STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)
Total wrong (compiled): 1548


DATASET: FOLIO

GPT-5: 433 wrong / 3514 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     88/598/609   -    34      -   8    33      -    12     1
Directed_T     53/607/609   -     -     30   9     -      -    14     -
Directed_F     49/604/609   9     -      -   -     -     32     -     8
  Nudged_T     48/606/609   -     -     16  13     -      -    19     -
  Nudged_F     45/602/609  10     -      -   -     -     18     -    17
 Two-Stage    150/497/609  11    23      -  10    16      -    10    80

DeepSeek-R1: 383 wrong / 3305 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     77/57

# Definite Wrong Cross-Condition Analysis

For cases that are **compiled + wrong + definite prediction (True/False)**:
- Distribution by GT and Pred
- Whether same case_idx was correct in other conditions (Hard vs Recoverable)

In [ ]:
# Definite Wrong Cross-Condition Analysis
# compiled + wrong + definite pred (T/F) - are they hard or recoverable by other conditions?

from collections import defaultdict, Counter

def norm_pred(p):
    p = str(p).lower()
    if p in ["yes"]: return "True"
    if p in ["no"]: return "False"
    if p in ["uncertain", "unknown", "failure"]: return "Unc"
    return p.capitalize()

def norm_gt(g):
    g = str(g).lower()
    if g in ["yes"]: return "True"
    if g in ["no"]: return "False"
    if g in ["uncertain", "unknown"]: return "Unc"
    return g.capitalize()

CONDS = [("Baseline", "baseline"), ("Directed_T", "bidir_true"), ("Directed_F", "bidir_false"),
         ("Nudged_T", "spooky_true"), ("Nudged_F", "spooky_false")]

for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        print(f"\n{'='*100}")
        print(f"{dataset} / {model}: DEFINITE WRONG CROSS-CONDITION ANALYSIS")
        print(f"{'='*100}")
        
        # Collect: results[run][case_idx][cond] = {compiled, correct, pred, gt}
        results = defaultdict(lambda: defaultdict(dict))
        for cond_name, cond in CONDS:
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries: continue
                for e in entries:
                    case_idx = e.get("case_idx")
                    compiled = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    results[run][case_idx][cond_name] = {
                        "compiled": compiled, "correct": e.get("correct", False),
                        "pred": norm_pred(e.get("prediction", "")), "gt": norm_gt(e.get("ground_truth", ""))
                    }
        
        # Analyze definite wrong cases
        rows = []  # (run, case_idx, cond, gt, pred, is_hard, recovered_by)
        for run, cases in results.items():
            for case_idx, conds in cases.items():
                for cond_name, res in conds.items():
                    if not res["compiled"] or res["correct"]: continue
                    if res["pred"] not in ["True", "False"]: continue  # skip conservative
                    
                    # Check recovery by other conditions
                    recovered_by = [c for c, r in conds.items() if c != cond_name and r["compiled"] and r["correct"]]
                    rows.append({
                        "run": run, "case_idx": case_idx, "cond": cond_name,
                        "gt": res["gt"], "pred": res["pred"],
                        "is_hard": len(recovered_by) == 0, "recovered_by": recovered_by
                    })
        
        df = pd.DataFrame(rows)
        if df.empty:
            print("No definite wrong cases found.")
            continue
        
        # Table 1: Distribution by GT→Pred
        print(f"\n1. Distribution by GT→Pred (n={len(df)})")
        print("-" * 60)
        ct = pd.crosstab(df["gt"], df["pred"], margins=True)
        print(ct.to_string())
        
        # Table 2: Hard vs Recoverable by condition
        print(f"\n2. Hard vs Recoverable by Condition")
        print("-" * 80)
        print(f"{'Condition':<12} {'Total':>8} {'Hard':>8} {'Hard%':>8} {'Recov':>8} {'Recov%':>8}")
        for cond_name, _ in CONDS:
            cdf = df[df["cond"] == cond_name]
            if len(cdf) == 0: continue
            hard = cdf["is_hard"].sum()
            recov = len(cdf) - hard
            print(f"{cond_name:<12} {len(cdf):>8} {hard:>8} {hard/len(cdf)*100:>7.1f}% {recov:>8} {recov/len(cdf)*100:>7.1f}%")
        
        # Table 3: Which conditions recover which?
        print(f"\n3. Recovery Matrix (row=wrong in, col=recovered by)")
        print("-" * 80)
        cond_names = [c[0] for c in CONDS]
        matrix = {c1: {c2: 0 for c2 in cond_names} for c1 in cond_names}
        for _, row in df.iterrows():
            for rc in row["recovered_by"]:
                matrix[row["cond"]][rc] += 1
        
        print(f"{'':>12}", end="")
        for c in cond_names: print(f" {c[:8]:>8}", end="")
        print()
        for c1 in cond_names:
            print(f"{c1:<12}", end="")
            for c2 in cond_names:
                v = matrix[c1][c2] if c1 != c2 else "-"
                print(f" {v:>8}", end="")
            print()

print("\nLegend: Hard=wrong in ALL conditions, Recov=correct in at least 1 other condition")

In [ ]:
# Multi-Stage Prediction Flow Sankey
import matplotlib.pyplot as plt
from collections import defaultdict

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not installed - using matplotlib fallback")

def get_pred_category(e, is_twostage=False):
    """Get prediction category: T, F, Unc, or NC (not compiled)."""
    if is_twostage:
        compiled = e.get("stage2_success", False)
    else:
        compiled = e.get("success") or e.get("lean_verification", {}).get("success", False)
    
    if not compiled:
        return "NC"
    
    pred = str(e.get("prediction", "")).lower()
    if pred in ["true", "yes"]: return "T"
    if pred in ["false", "no"]: return "F"
    return "Unc"

def get_gt_category(e):
    """Get ground truth category: T, F, Unc."""
    gt = str(e.get("ground_truth", "")).lower()
    if gt in ["true", "yes"]: return "T"
    if gt in ["false", "no"]: return "F"
    return "Unc"

# Define stages for T and F directions
T_STAGES = [("GT", None), ("Baseline", "baseline"), ("Directed_T", "bidir_true"), 
            ("Nudged_T", "spooky_true"), ("Two-Stage", "twostage")]
F_STAGES = [("GT", None), ("Baseline", "baseline"), ("Directed_F", "bidir_false"), 
            ("Nudged_F", "spooky_false"), ("Two-Stage", "twostage")]

CATEGORIES = ["T", "F", "Unc", "NC"]
CAT_COLORS = {"T": "#2ecc71", "F": "#e74c3c", "Unc": "#f39c12", "NC": "#95a5a6"}

def collect_stage_data(dataset_dir, model_dir, stages, run=1):
    """Collect predictions for each case across all stages."""
    # First, get GT and case_idx from baseline
    baseline_entries = load_run(f"../results/{dataset_dir}/{model_dir}/baseline_run{run}/results.jsonl")
    if not baseline_entries:
        return {}
    
    case_data = {}
    for e in baseline_entries:
        case_idx = e.get("case_idx")
        case_data[case_idx] = {"GT": get_gt_category(e)}
    
    # Collect predictions for each stage
    for stage_name, stage_dir in stages[1:]:  # Skip GT
        if stage_dir == "twostage":
            model_name = "deepseek-r1" if "deepseek" in model_dir else "gpt-5"
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            is_twostage = True
        else:
            entries = load_run(f"../results/{dataset_dir}/{model_dir}/{stage_dir}_run{run}/results.jsonl")
            is_twostage = False
        
        if not entries:
            continue
        
        for e in entries:
            case_idx = e.get("case_idx")
            if case_idx in case_data:
                case_data[case_idx][stage_name] = get_pred_category(e, is_twostage)
    
    return case_data

def compute_flows(case_data, stages):
    """Compute flows between consecutive stages."""
    stage_names = [s[0] for s in stages]
    flows = {}  # (stage_from, stage_to) -> {(cat_from, cat_to): count}
    
    for i in range(len(stage_names) - 1):
        s_from, s_to = stage_names[i], stage_names[i+1]
        flows[(s_from, s_to)] = defaultdict(int)
        
        for case_idx, data in case_data.items():
            if s_from in data and s_to in data:
                cat_from = data[s_from]
                cat_to = data[s_to]
                flows[(s_from, s_to)][(cat_from, cat_to)] += 1
    
    return flows

def create_sankey_plotly(flows, stages, title):
    """Create Sankey diagram using Plotly."""
    stage_names = [s[0] for s in stages]
    
    # Create node labels: each stage has 4 categories
    node_labels = []
    node_colors = []
    for stage in stage_names:
        for cat in CATEGORIES:
            node_labels.append(f"{stage}:{cat}")
            node_colors.append(CAT_COLORS[cat])
    
    # Create links
    sources, targets, values, link_colors = [], [], [], []
    
    for i, (s_from, s_to) in enumerate(zip(stage_names[:-1], stage_names[1:])):
        flow_data = flows.get((s_from, s_to), {})
        for (cat_from, cat_to), count in flow_data.items():
            if count > 0:
                # Node indices
                src_idx = i * 4 + CATEGORIES.index(cat_from)
                tgt_idx = (i + 1) * 4 + CATEGORIES.index(cat_to)
                sources.append(src_idx)
                targets.append(tgt_idx)
                values.append(count)
                # Color based on source category with transparency
                color = CAT_COLORS[cat_from]
                r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
                link_colors.append(f"rgba({r},{g},{b},0.4)")
    
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    )])
    
    fig.update_layout(title_text=title, font_size=10, height=500, width=1000)
    return fig

# Generate Sankey diagrams for each dataset/model combination
for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        print(f"\n{'='*80}")
        print(f"{dataset} / {model}")
        print(f"{'='*80}")
        
        # Aggregate across all 3 runs
        t_flows_agg = defaultdict(lambda: defaultdict(int))
        f_flows_agg = defaultdict(lambda: defaultdict(int))
        
        for run in [1, 2, 3]:
            # T-direction
            t_data = collect_stage_data(dataset_dir, model_dir, T_STAGES, run)
            t_flows = compute_flows(t_data, T_STAGES)
            for key, flow_dict in t_flows.items():
                for pair, count in flow_dict.items():
                    t_flows_agg[key][pair] += count
            
            # F-direction
            f_data = collect_stage_data(dataset_dir, model_dir, F_STAGES, run)
            f_flows = compute_flows(f_data, F_STAGES)
            for key, flow_dict in f_flows.items():
                for pair, count in flow_dict.items():
                    f_flows_agg[key][pair] += count
        
        if HAS_PLOTLY:
            # T-direction Sankey
            fig_t = create_sankey_plotly(dict(t_flows_agg), T_STAGES, 
                                         f"{dataset}/{model}: T-Direction (GT→Baseline→Dir_T→Nudge_T→TwoStage)")
            fig_t.write_html(f"../results/llm-as-judge/sankey_T_{dataset}_{model_dir}.html")
            print(f"Saved: sankey_T_{dataset}_{model_dir}.html")
            
            # F-direction Sankey
            fig_f = create_sankey_plotly(dict(f_flows_agg), F_STAGES,
                                         f"{dataset}/{model}: F-Direction (GT→Baseline→Dir_F→Nudge_F→TwoStage)")
            fig_f.write_html(f"../results/llm-as-judge/sankey_F_{dataset}_{model_dir}.html")
            print(f"Saved: sankey_F_{dataset}_{model_dir}.html")
        
        # Print summary tables
        print(f"\nT-Direction Flow Summary (aggregated over 3 runs):")
        t_stage_names = [s[0] for s in T_STAGES]
        for i in range(len(t_stage_names) - 1):
            key = (t_stage_names[i], t_stage_names[i+1])
            print(f"  {key[0]} → {key[1]}:")
            flow_dict = t_flows_agg.get(key, {})
            for (c1, c2), count in sorted(flow_dict.items(), key=lambda x: -x[1])[:5]:
                print(f"    {c1}→{c2}: {count}")
        
        print(f"\nF-Direction Flow Summary (aggregated over 3 runs):")
        f_stage_names = [s[0] for s in F_STAGES]
        for i in range(len(f_stage_names) - 1):
            key = (f_stage_names[i], f_stage_names[i+1])
            print(f"  {key[0]} → {key[1]}:")
            flow_dict = f_flows_agg.get(key, {})
            for (c1, c2), count in sorted(flow_dict.items(), key=lambda x: -x[1])[:5]:
                print(f"    {c1}→{c2}: {count}")

In [ ]:
# Merged Sankey: T and F directions combined
# Shows GT → Baseline → [Directed_T & Directed_F] → [Nudged_T & Nudged_F] → Two-Stage

def collect_merged_stage_data(dataset_dir, model_dir, run=1):
    """Collect predictions with both T and F directions."""
    baseline_entries = load_run(f"../results/{dataset_dir}/{model_dir}/baseline_run{run}/results.jsonl")
    if not baseline_entries:
        return {}
    
    case_data = {}
    for e in baseline_entries:
        case_idx = e.get("case_idx")
        case_data[case_idx] = {
            "GT": get_gt_category(e),
            "Baseline": get_pred_category(e)
        }
    
    # Directed T and F
    for stage_name, stage_dir in [("Dir_T", "bidir_true"), ("Dir_F", "bidir_false")]:
        entries = load_run(f"../results/{dataset_dir}/{model_dir}/{stage_dir}_run{run}/results.jsonl")
        if entries:
            for e in entries:
                case_idx = e.get("case_idx")
                if case_idx in case_data:
                    case_data[case_idx][stage_name] = get_pred_category(e)
    
    # Nudged T and F
    for stage_name, stage_dir in [("Nudge_T", "spooky_true"), ("Nudge_F", "spooky_false")]:
        entries = load_run(f"../results/{dataset_dir}/{model_dir}/{stage_dir}_run{run}/results.jsonl")
        if entries:
            for e in entries:
                case_idx = e.get("case_idx")
                if case_idx in case_data:
                    case_data[case_idx][stage_name] = get_pred_category(e)
    
    # Two-Stage
    model_name = "deepseek-r1" if "deepseek" in model_dir else "gpt-5"
    entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
    if entries:
        for e in entries:
            case_idx = e.get("case_idx")
            if case_idx in case_data:
                case_data[case_idx]["TwoStage"] = get_pred_category(e, is_twostage=True)
    
    return case_data

def create_merged_sankey(dataset_dir, model_dir, title):
    """Create merged Sankey with both T and F branches."""
    # Aggregate across runs
    all_data = defaultdict(dict)
    for run in [1, 2, 3]:
        case_data = collect_merged_stage_data(dataset_dir, model_dir, run)
        for case_idx, data in case_data.items():
            for stage, val in data.items():
                all_data[(run, case_idx)][stage] = val
    
    # Node structure: GT(4) -> Baseline(4) -> Dir_T(4) + Dir_F(4) -> Nudge_T(4) + Nudge_F(4) -> TwoStage(4)
    # Total nodes: 4 + 4 + 8 + 8 + 4 = 28
    stages = ["GT", "Baseline", "Dir_T", "Dir_F", "Nudge_T", "Nudge_F", "TwoStage"]
    node_labels = []
    node_colors = []
    
    for stage in stages:
        for cat in CATEGORIES:
            node_labels.append(f"{stage}:{cat}")
            node_colors.append(CAT_COLORS[cat])
    
    # Node index mapping
    def get_node_idx(stage, cat):
        stage_idx = stages.index(stage)
        return stage_idx * 4 + CATEGORIES.index(cat)
    
    # Compute flows
    flows = defaultdict(int)  # (src_idx, tgt_idx) -> count
    
    for key, data in all_data.items():
        gt = data.get("GT")
        baseline = data.get("Baseline")
        dir_t = data.get("Dir_T")
        dir_f = data.get("Dir_F")
        nudge_t = data.get("Nudge_T")
        nudge_f = data.get("Nudge_F")
        twostage = data.get("TwoStage")
        
        # GT -> Baseline
        if gt and baseline:
            flows[(get_node_idx("GT", gt), get_node_idx("Baseline", baseline))] += 1
        
        # Baseline -> Dir_T and Dir_F
        if baseline and dir_t:
            flows[(get_node_idx("Baseline", baseline), get_node_idx("Dir_T", dir_t))] += 1
        if baseline and dir_f:
            flows[(get_node_idx("Baseline", baseline), get_node_idx("Dir_F", dir_f))] += 1
        
        # Dir_T -> Nudge_T, Dir_F -> Nudge_F
        if dir_t and nudge_t:
            flows[(get_node_idx("Dir_T", dir_t), get_node_idx("Nudge_T", nudge_t))] += 1
        if dir_f and nudge_f:
            flows[(get_node_idx("Dir_F", dir_f), get_node_idx("Nudge_F", nudge_f))] += 1
        
        # Nudge_T -> TwoStage, Nudge_F -> TwoStage
        if nudge_t and twostage:
            flows[(get_node_idx("Nudge_T", nudge_t), get_node_idx("TwoStage", twostage))] += 1
        if nudge_f and twostage:
            flows[(get_node_idx("Nudge_F", nudge_f), get_node_idx("TwoStage", twostage))] += 1
    
    sources, targets, values, link_colors = [], [], [], []
    for (src, tgt), count in flows.items():
        if count > 0:
            sources.append(src)
            targets.append(tgt)
            values.append(count)
            # Color based on source node
            src_cat = node_labels[src].split(":")[1]
            color = CAT_COLORS[src_cat]
            r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
            link_colors.append(f"rgba({r},{g},{b},0.4)")
    
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15, thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels, color=node_colors
        ),
        link=dict(
            source=sources, target=targets, value=values, color=link_colors
        )
    )])
    
    fig.update_layout(title_text=title, font_size=10, height=600, width=1200)
    return fig

# Generate merged Sankey diagrams
if HAS_PLOTLY:
    for dataset, dataset_dir in DATASETS:
        for model, model_dir in MODELS:
            fig = create_merged_sankey(dataset_dir, model_dir, 
                                       f"{dataset}/{model}: Merged Flow (T & F directions)")
            fig.write_html(f"../results/llm-as-judge/sankey_merged_{dataset}_{model_dir}.html")
            print(f"Saved: sankey_merged_{dataset}_{model_dir}.html")
else:
    print("Plotly not available - skipping merged Sankey")

# Prediction Flow Sankey Diagrams

Multi-stage Sankey showing how predictions flow across conditions for the same cases:
- **T-Direction**: GT → Baseline → Directed_T → Nudged_T → Two-Stage
- **F-Direction**: GT → Baseline → Directed_F → Nudged_F → Two-Stage
- **Merged**: Both directions combined